In [2]:
#import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

In [24]:
#get data
doners_data = pd.read_csv("/Users/sararedaelli/Desktop/transconjugacy efficacy PROJECT/data_donatori.csv")
transconjugants_data = pd.read_csv("/Users/sararedaelli/Desktop/transconjugacy efficacy PROJECT/data.csv")

doners_data.head()
transconjugants_data.head()

,Idx Experiment,Idx Replica Diluizione,Idx Replica Condizione,Controllo,Diluition,IBU,DMSO,Temp,Conta
0,1,1,A,No,-1,6.25,0,28,TMTC
1,1,2,A,No,-1,6.25,0,28,TMTC
2,1,3,A,No,-1,6.25,0,28,TMTC
3,1,1,A,No,-2,6.25,0,28,36
4,1,2,A,No,-2,6.25,0,28,33


In [25]:
col_map = {"Idx Experiment": "i", "Idx Replica Diluizione": "k", "Controllo": "c", "Idx Replica Condizione": "j","Conta": "Count", "Diluition": "Dilution"}
transconjugants_data.rename(columns=col_map, inplace=True)
doners_data.rename(columns=col_map, inplace=True)
# transconjugants_data = transconjugants_data[~transconjugants_data['i'].astype(str).isin(['5','6','7', '8'])].copy()

transconjugants_data = transconjugants_data[~transconjugants_data['i'].astype(str).isin(['5,6','7,8'])].copy()
# transconjugants_data = transconjugants_data[~transconjugants_data['Count'].astype(str).isin(['TMTC'])].copy()
TMTC = 1000
transconjugants_data['Count'] = transconjugants_data['Count'].replace('TMTC', TMTC)

transconjugants_data['c'] = transconjugants_data['c'].replace('No', -1)
transconjugants_data['c'] = transconjugants_data['c'].replace('Yes', 1)

transconjugants_data['j'] = transconjugants_data['j'].replace('A', 0)
transconjugants_data['j'] = transconjugants_data['j'].replace('B', 1)
transconjugants_data['j'] = transconjugants_data['j'].replace('C', 2)


doners_data = (
    doners_data
        .copy()
        # make sure it's a clean string, then split on comma
        .assign(i=lambda df: df['i'].astype(str).str.replace(' ', '').str.split(','))
        .explode('i')          # "5,6" -> ["5","6"] -> two rows
)


for col in transconjugants_data.columns:
    # if col == "i":
    #     continue
    transconjugants_data[col] = pd.to_numeric(transconjugants_data[col], errors='coerce').dropna()
    
for col in doners_data.columns:
    # if col == "i":
    #     continue
    doners_data[col] = pd.to_numeric(doners_data[col], errors='coerce').dropna()
    
transconjugants_data['Final_count'] = transconjugants_data["Count"] * (10 ** (-transconjugants_data['Dilution']))
doners_data['Final_count'] = doners_data["Count"] * (10 ** (-doners_data['Dilution']))



/var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/ipykernel_46449/3304602194.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  transconjugants_data['c'] = transconjugants_data['c'].replace('Yes', 1)
/var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/ipykernel_46449/3304602194.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  transconjugants_data['j'] = transconjugants_data['j'].replace('C', 2)


In [27]:
transconjugants_data[0:18]
#I want to change the values in k counting from 1 to 9 rows with the same i, j and c
transconjugants_data['k'] = transconjugants_data.groupby(['i', 'j', 'c']).cumcount() + 1
#doners_data['k'] = doners_data.groupby(['i', 'j']).cumcount() + 1
transconjugants_data[0:18]
#I want to have column j before colomun k
transconjugants_data[0:18]
transconjugants_data = transconjugants_data[['i', 'j', 'k', 'c', 'Dilution', 'Count', 'Final_count']]
transconjugants_data[0:18]

,i,j,k,c,Dilution,Count,Final_count
0,1,0,1,-1,-1,1000,10000
1,1,0,2,-1,-1,1000,10000
2,1,0,3,-1,-1,1000,10000
3,1,0,4,-1,-2,36,3600
4,1,0,5,-1,-2,33,3300
5,1,0,6,-1,-2,30,3000
6,1,0,7,-1,-3,4,4000
7,1,0,8,-1,-3,6,6000
8,1,0,9,-1,-3,19,19000
9,1,0,1,1,-1,1000,10000
